# Variational Classifier — Iris Dataset


## Librerias

In [1]:
# 1. Import all libraries
import pandas as pd
import time
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer
from sklearn import datasets
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)


## Data

In [2]:
data = pd.DataFrame(datasets.load_iris().data, columns=datasets.load_iris().feature_names)
data['target'] = datasets.load_iris().target
data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [3]:
data.target.unique()

array([0, 1, 2])

## Data Preparation

In [4]:
n_qubits = 4

X = data.drop(columns='target')
y = data['target']

# Split test set with a ratio of 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Custom Dataset Loaded: Train={len(X_train)}, Test={len(X_test)}")

# Normalize to [0, π]
minmax = MinMaxScaler(feature_range=(0, np.pi))
X_train = minmax.fit_transform(X_train)
X_test  = minmax.transform(X_test)
X_test  = np.clip(X_test, 0, np.pi)

print(f"Final shapes -> Train: {X_train.shape}, Test: {X_test.shape}")

Custom Dataset Loaded: Train=120, Test=30
Final shapes -> Train: (120, 4), Test: (30, 4)


## Variational Classifier

In [5]:
dev = qml.device("default.qubit")

### Variational Data Prep

In [6]:
X = np.array(X, dtype=float)
y = np.array(y)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(X)

In [7]:
def layer(layer_weights):
    for wire in range(4):
        qml.Rot(*layer_weights[wire], wires=wire)
    for wires in ([0, 1], [1, 2], [2, 3], [3, 0]):
        qml.CNOT(wires=wires)

In [8]:
def state_preparation(x):
    for i in range(4):
        qml.RY(x[i], wires=i)

In [9]:
@qml.qnode(dev)
def circuit(weights, x):
    state_preparation(x)
    for layer_weights in weights:
        layer(layer_weights)
    return [
        qml.expval(qml.PauliZ(0)),
        qml.expval(qml.PauliZ(1)),
        qml.expval(qml.PauliZ(2)),
    ]

def variational_classifier(weights, bias, x):
    out = qml.math.stack(circuit(weights, x))
    return out + bias

def cost(weights, bias, X, Y):
    predictions = qml.math.stack(
        [variational_classifier(weights, bias, x) for x in X]
    )
    Y = np.array(Y, dtype=int)
    targets = np.eye(3)[Y]
    return qml.math.mean((predictions - targets) ** 2)

In [10]:
# --- Tiempo de entrenamiento ---
np.random.seed(0)
weights = 0.01 * np.random.randn(2, 4, 3, requires_grad=True)
bias = np.zeros(3, requires_grad=True)
opt = NesterovMomentumOptimizer(0.1)
batch_size = 5

start = time.time()
for it in range(100):
    batch_index = np.random.randint(0, len(X), (batch_size,))
    X_batch = X[batch_index]
    Y_batch = y[batch_index]
    weights, bias = opt.step(cost, weights, bias, X=X_batch, Y=Y_batch)

    predictions = [np.argmax(variational_classifier(weights, bias, x)) for x in X]
    current_cost = cost(weights, bias, X, y)
    acc = np.mean(np.array(predictions) == np.array(y))
    print(f"Iter: {it+1:4d} | Cost: {current_cost:0.7f} | Accuracy: {acc:0.7f}")
train_time = time.time() - start
print(f"\nTraining time: {train_time:.4f} s")


Iter:    1 | Cost: 0.5188235 | Accuracy: 0.1133333
Iter:    2 | Cost: 0.4869285 | Accuracy: 0.0933333
Iter:    3 | Cost: 0.4519782 | Accuracy: 0.1266667
Iter:    4 | Cost: 0.4219330 | Accuracy: 0.1666667
Iter:    5 | Cost: 0.3959456 | Accuracy: 0.2000000
Iter:    6 | Cost: 0.3759545 | Accuracy: 0.2466667
Iter:    7 | Cost: 0.3600332 | Accuracy: 0.2933333
Iter:    8 | Cost: 0.3479469 | Accuracy: 0.3000000
Iter:    9 | Cost: 0.3356837 | Accuracy: 0.3066667
Iter:   10 | Cost: 0.3218714 | Accuracy: 0.3133333
Iter:   11 | Cost: 0.3049537 | Accuracy: 0.2933333
Iter:   12 | Cost: 0.2872706 | Accuracy: 0.3133333
Iter:   13 | Cost: 0.2714625 | Accuracy: 0.3800000
Iter:   14 | Cost: 0.2548136 | Accuracy: 0.4466667
Iter:   15 | Cost: 0.2353031 | Accuracy: 0.5133333
Iter:   16 | Cost: 0.2178201 | Accuracy: 0.5533333
Iter:   17 | Cost: 0.2028099 | Accuracy: 0.5666667
Iter:   18 | Cost: 0.1933868 | Accuracy: 0.5800000
Iter:   19 | Cost: 0.1871939 | Accuracy: 0.6133333
Iter:   20 | Cost: 0.1827295 | 

In [11]:
# Predicción
def predict(weights, bias, X):
    return np.array([
        np.argmax(variational_classifier(weights, bias, x))
        for x in X
    ])

        # --- Tiempo de predicción (inferencia) ---
start = time.time()
y_test_pred = predict(weights, bias, X_test)
infer_time = time.time() - start

acc_test = float(np.mean(np.array(y_test_pred) == np.array(y_test)))
print("Test accuracy:", acc_test)
print(f"Inference time: {infer_time:.4f} s")
print(f"Training time:  {train_time:.4f} s")

Test accuracy: 0.8666666666666667
Inference time: 0.0755 s
Training time:  73.7743 s


In [12]:
y_test_pred = predict(weights, bias, X_test)
acc_test = float(np.mean(np.array(y_test_pred) == np.array(y_test)))
print("Test accuracy:", acc_test)

Test accuracy: 0.8666666666666667
